# Notebook: análise de variáveis do CadÚnico

id_rastreabilidade: RTB_003  
Versão: v01  
Data de criação: 09/04/2026  
Última atualização: 09/04/2026  
Responsável: Ailton Vendramini  

---

## 🎯 Finalidade

Analisar e validar as variáveis do IVS-H deriváveis da base CadÚnico, identificando regras de construção, consistência e limitações dos dados, com foco na preparação para o cálculo do índice.

---

## 📥 Fonte de Dados

Fonte: CadÚnico  
Camada de entrada: 02_limpo (ou 01_bruto, conforme necessidade de validação)  
Camada de saída: 03_curado  

Período de referência da execução: 2025_12  
Período de comparação: —  
Tipo de recorte temporal: corte transversal  

Arquivo de entrada:
../../dados/cadunico/02_limpo/2025_12/cadunico_limpo.parquet  

---

## 🧱 Base Conceitual

Documentos de apoio conceitual e metodológico do projeto:

- docs/metodologia_ivsh.md  
- docs/regras_negocio.md  
- docs/padrao_nomenclatura.md  

---

## ⚙️ Escopo técnico deste notebook

Este notebook deve:

- identificar variáveis do IVS-H disponíveis na base CadÚnico  
- analisar estrutura e qualidade das variáveis selecionadas  
- testar regras de construção das variáveis (ex: RT_01, CH_06, etc.)  
- identificar inconsistências e limitações dos dados  
- gerar outputs intermediários para validação  

Este notebook não deve:

- calcular o IVS-H final  
- aplicar pesos do índice  
- misturar variáveis de outras fontes sem documentação  
- sobrescrever dados da camada 02_limpo  

---

## 📊 Outputs esperados

| tipo_output  | descrição                               | destino                         |
|--------------|------------------------------------------|----------------------------------|
| exploratorio | análise de variáveis individuais         | não salvar                      |
| analitico    | tabelas intermediárias por variável      | outputs/tabelas/cadunico/       |
| operacional  | base estruturada por variável (curada)   | dados/cadunico/03_curado/       |

---

## 🧠 Observações

- A base apresenta inconsistências de nomenclatura territorial (ex: "JARDIM AMANDA 2" vs "JARDIM AMANDA II"), que podem impactar análises por território.  
- As variáveis devem ser analisadas considerando possíveis duplicidades semânticas.  
- Este notebook é etapa intermediária entre tratamento e cálculo do índice.  
- Ajustes relevantes devem ser registrados para futura padronização na dimensão territorial.  

## 🗄️ Fontes de dados e estratégia de integração

A construção do IVS-H utiliza, nesta etapa inicial, a base do CadÚnico como principal fonte administrativa.

Adicionalmente, dados do IVS do :contentReference[oaicite:0]{index=0} foram consultados via ambiente analítico (BigQuery), permitindo acesso estruturado às informações municipais derivadas do Censo Demográfico de 2010.

Essas informações são utilizadas como referência histórica e comparativa, não como base direta de cálculo nesta etapa.

---

## 🔗 Integração futura de bases

O projeto prevê a incorporação progressiva de outras fontes de dados, tais como:

- microdados do :contentReference[oaicite:1]{index=1} (em fase de disponibilização)
- bases estaduais, como a :contentReference[oaicite:2]{index=2}
- registros administrativos complementares (ex: trabalho, educação, saúde)

Essa estratégia permitirá:

- ampliar a cobertura das variáveis do IVS-H
- reduzir a dependência de proxies operacionais
- aumentar a robustez analítica do modelo

---

## ⚠️ Delimitação desta etapa

Apesar da disponibilidade de dados históricos e comparativos, este notebook mantém foco exclusivo na análise das variáveis deriváveis da base CadÚnico.

A incorporação de outras bases será realizada em etapas posteriores, de forma estruturada, rastreável e documentada.

## ⚠️ Nota sobre o universo do CadÚnico

O Cadastro Único é um instrumento de identificação e caracterização socioeconômica das famílias, e não um filtro simples de renda.

Nele, podem constar:

- famílias com renda mensal de até 1/2 salário mínimo por pessoa, que constituem o público prioritário;
- famílias com renda superior a esse limite, desde que o cadastramento esteja vinculado à inclusão ou ao acompanhamento de programas, benefícios ou serviços específicos.

Dessa forma, o universo analisado neste notebook deve ser interpretado como uma base administrativa de políticas públicas, e não como um recorte exclusivo da população em situação de extrema vulnerabilidade.

Essa distinção é fundamental para a correta interpretação da variável RT_01.

In [1]:
import pandas as pd

## 📘 Dicionário de dados — referência

O dicionário completo encontra-se em:

docs/dicionario_v02.md

Trecho relevante para esta análise:

[cole apenas o trecho necessário]

dicionario = pd.read_csv(r"D:/Ailton/PMH/Cidades_Inteligentes/inclusão/cadúnico_projeto/cadunico_mvp_sqlite/tratativas e informações/tutorial_para_utilização_cadúnico_software/ECAD dezembro 2025 com PBF/dicionariotudo.csv")
dicionario.head()

In [2]:
import pandas as pd

caminho = "../../dados/cadunico/01_bruto/2025_12/cadunico.csv"

df = pd.read_csv(caminho, sep=';', encoding='latin1')

df.head()

C:\Users\ailto\AppData\Local\Temp\ipykernel_27116\2575082178.py:5: DtypeWarning: Columns (40,73,82) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(caminho, sep=';', encoding='latin1')


,d.cd_ibge,d.cod_familiar_fam,d.dat_cadastramento_fam,d.dat_atual_fam,d.cod_est_cadastral_fam,d.cod_forma_coleta_fam,d.dta_entrevista_fam,d.nom_localidade_fam,d.nom_tip_logradouro_fam,d.nom_titulo_logradouro_fam,...,p.ind_dinh_carregador_memb,p.ind_dinh_catador_memb,p.ind_dinh_servs_gerais_memb,p.ind_dinh_pede_memb,p.ind_dinh_vendas_memb,p.ind_dinh_outro_memb,p.ind_dinh_nao_resp_memb,p.ind_atend_nenhum_memb,p.ref_cad,p.ref_pbf
0,3519071,8477396,17/12/2001,04/08/2025,3,1,04/08/2025,JARDIM MINDA,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0
1,3519071,8477396,17/12/2001,04/08/2025,3,1,04/08/2025,JARDIM MINDA,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0
2,3519071,8477396,17/12/2001,04/08/2025,3,1,04/08/2025,JARDIM MINDA,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0
3,3519071,8477396,17/12/2001,04/08/2025,3,1,04/08/2025,JARDIM MINDA,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0
4,3519071,37372840,07/02/2002,14/04/2025,3,1,14/04/2025,JARDIM MONTE SINAI,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0


In [3]:
df.columns = df.columns.str.strip()

In [4]:
df.dtypes

d.cd_ibge                     int64
d.cod_familiar_fam            int64
d.dat_cadastramento_fam      object
d.dat_atual_fam              object
d.cod_est_cadastral_fam       int64
                             ...   
p.ind_dinh_outro_memb       float64
p.ind_dinh_nao_resp_memb    float64
p.ind_atend_nenhum_memb     float64
p.ref_cad                    object
p.ref_pbf                   float64
Length: 211, dtype: object

In [5]:
for col in df.columns:
    print(col)

d.cd_ibge
d.cod_familiar_fam
d.dat_cadastramento_fam
d.dat_atual_fam
d.cod_est_cadastral_fam
d.cod_forma_coleta_fam
d.dta_entrevista_fam
d.nom_localidade_fam
d.nom_tip_logradouro_fam
d.nom_titulo_logradouro_fam
d.nom_logradouro_fam
d.num_logradouro_fam
d.des_complemento_fam
d.des_complemento_adic_fam
d.num_cep_logradouro_fam
d.cod_unidade_territorial_fam
d.nom_unidade_territorial_fam
d.txt_referencia_local_fam
d.nom_entrevistador_fam
d.num_cpf_entrevistador_fam
d.vlr_renda_media_fam
d.fx_rfpc
d.vlr_renda_total_fam
d.marc_pbf
d.qtde_meses_desat_cat
d.cod_local_domic_fam
d.cod_especie_domic_fam
d.qtd_comodos_domic_fam
d.qtd_comodos_dormitorio_fam
d.cod_material_piso_fam
d.cod_material_domic_fam
d.cod_agua_canalizada_fam
d.cod_abaste_agua_domic_fam
d.cod_banheiro_domic_fam
d.cod_escoa_sanitario_domic_fam
d.cod_destino_lixo_domic_fam
d.cod_iluminacao_domic_fam
d.cod_calcamento_domic_fam
d.cod_familia_indigena_fam
d.cod_povo_indigena_fam
d.nom_povo_indigena_fam
d.cod_indigena_reside_fam
d.c

In [6]:
[col for col in df.columns if "renda" in col.lower()]

['d.vlr_renda_media_fam',
 'd.vlr_renda_total_fam',
 'p.fx_renda_individual_805',
 'p.fx_renda_individual_808',
 'p.fx_renda_individual_809_1',
 'p.fx_renda_individual_809_2',
 'p.fx_renda_individual_809_3',
 'p.fx_renda_individual_809_4',
 'p.fx_renda_individual_809_5']

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72424 entries, 0 to 72423
Columns: 211 entries, d.cd_ibge to p.ref_pbf
dtypes: float64(134), int64(31), object(46)
memory usage: 116.6+ MB


In [8]:
for col in df.columns:
    if "renda" in col.lower():
        print(col)

d.vlr_renda_media_fam
d.vlr_renda_total_fam
p.fx_renda_individual_805
p.fx_renda_individual_808
p.fx_renda_individual_809_1
p.fx_renda_individual_809_2
p.fx_renda_individual_809_3
p.fx_renda_individual_809_4
p.fx_renda_individual_809_5


In [9]:
[col for col in df.columns if "qtd" in col.lower()]

['d.qtde_meses_desat_cat',
 'd.qtd_comodos_domic_fam',
 'd.qtd_comodos_dormitorio_fam',
 'd.qtd_pessoas_domic_fam',
 'd.qtd_familias_domic_fam',
 'd.qtd_pessoa_inter_0_17_anos_fam',
 'd.qtd_pessoa_inter_18_64_anos_fam',
 'd.qtd_pessoa_inter_65_anos_fam',
 'p.qtd_meses_12_meses_memb',
 'p.qtd_dormir_freq_rua_memb',
 'p.qtd_dormir_freq_albergue_memb',
 'p.qtd_dormir_freq_dom_part_memb',
 'p.qtd_freq_outro_memb']

In [10]:
[col for col in df.columns if "membro" in col.lower()]

[]

In [11]:
[col for col in df.columns if "pessoa" in col.lower()]

['d.qtd_pessoas_domic_fam',
 'd.qtd_pessoa_inter_0_17_anos_fam',
 'd.qtd_pessoa_inter_18_64_anos_fam',
 'd.qtd_pessoa_inter_65_anos_fam',
 'p.ind_trabalho_infantil_pessoa',
 'p.nom_pessoa',
 'p.num_nis_pessoa_atual',
 'p.nom_apelido_pessoa',
 'p.cod_sexo_pessoa',
 'p.dta_nasc_pessoa',
 'p.cod_parentesco_rf_pessoa',
 'p.cod_raca_cor_pessoa',
 'p.nom_completo_mae_pessoa',
 'p.nom_completo_pai_pessoa',
 'p.cod_local_nascimento_pessoa',
 'p.sig_uf_munic_nasc_pessoa',
 'p.nom_ibge_munic_nasc_pessoa',
 'p.cod_ibge_munic_nasc_pessoa',
 'p.nom_pais_origem_pessoa',
 'p.cod_pais_origem_pessoa',
 'p.cod_certidao_registrada_pessoa',
 'p.cod_certidao_civil_pessoa',
 'p.cod_livro_termo_certid_pessoa',
 'p.cod_folha_termo_certid_pessoa',
 'p.cod_termo_matricula_certid_pessoa',
 'p.nom_munic_certid_pessoa',
 'p.cod_ibge_munic_certid_pessoa',
 'p.cod_cartorio_certid_pessoa',
 'p.num_cpf_pessoa',
 'p.num_identidade_pessoa',
 'p.cod_complemento_pessoa',
 'p.dta_emissao_ident_pessoa',
 'p.sig_uf_ident_pes

## 📊 RT_01 — Renda domiciliar per capita ≤ 1/2 salário mínimo

### 🎯 Objetivo

Identificar a proporção de pessoas em domicílios com renda domiciliar per capita inferior ou igual a meio salário mínimo, conforme definição utilizada no modelo do IVS.

---

### 🧮 Definição conceitual

A variável RT_01 busca capturar a condição de vulnerabilidade associada à baixa renda domiciliar, considerando a capacidade econômica média por indivíduo dentro do domicílio.

Formalmente:

Renda per capita = Renda total da família / Número de pessoas no domicílio

---

### 📥 Variáveis utilizadas (CadÚnico)

Para operacionalização desta variável, foram utilizadas as seguintes colunas:

- `d.vlr_renda_total_fam` → valor da renda total da família  
- `d.qtd_pessoas_domic_fam` → número de pessoas no domicílio  

---

### 📏 Parâmetro de corte

Considerando a base de referência (dezembro de 2025):

- Salário mínimo: R$ 1.412  
- Limite (1/2 salário mínimo): R$ 706  

Serão considerados em situação de vulnerabilidade os indivíduos pertencentes a domicílios com renda per capita ≤ R$ 706.

---

### ⚠️ Observações metodológicas

- A renda utilizada refere-se à renda declarada no CadÚnico, podendo estar sujeita a subdeclaração.  
- O cálculo considera o domicílio como unidade de referência econômica.  
- Esta variável será posteriormente normalizada e integrada ao cálculo do IVS-H.  

---

### 🔎 Estratégia de cálculo

1. calcular renda per capita para cada domicílio  
2. identificar domicílios abaixo do limite de R$ 706  
3. calcular a proporção de indivíduos nessas condições  

### Nota metodológica sobre o universo do CadÚnico

O Cadastro Único não se restringe, necessariamente, a famílias com renda per capita inferior ou igual a 1/2 salário mínimo. Também podem constar famílias com renda superior, desde que vinculadas ou pleiteando programas específicos que utilizem essa base.

Assim, o RT_01 deve ser interpretado como uma variável analítica construída dentro do universo do CadÚnico, e não como descrição exaustiva de toda a população vulnerável do município.

In [12]:
# calcular renda per capita
df["renda_per_capita"] = df["d.vlr_renda_total_fam"] / df["d.qtd_pessoas_domic_fam"]

df["renda_per_capita"].head(40)

0      379.0
1      379.0
2      379.0
3      379.0
4        0.0
5        0.0
6      191.0
7      191.0
8      191.0
9      191.0
10       0.0
11       0.0
12       0.0
13     148.0
14     148.0
15     148.0
16     148.0
17      41.0
18      41.0
19      41.0
20      41.0
21    1518.0
22    1518.0
23       0.0
24       0.0
25       0.0
26       0.0
27     346.0
28     346.0
29     346.0
30      25.0
31      25.0
32      25.0
33      25.0
34     718.0
35       0.0
36       0.0
37       0.0
38    2216.0
39    2216.0
Name: renda_per_capita, dtype: float64

In [13]:
# quantidade de famílias com renda total igual a zero
(df["d.vlr_renda_total_fam"] == 0).sum()

13568

In [14]:
# percentual
total = len(df)
zero = (df["d.vlr_renda_total_fam"] == 0).sum()

zero, zero / total

(13568, 0.18734121285761626)

In [15]:
familias_zero = df[df["d.vlr_renda_total_fam"] == 0].copy()

familias_zero.head()

,d.cd_ibge,d.cod_familiar_fam,d.dat_cadastramento_fam,d.dat_atual_fam,d.cod_est_cadastral_fam,d.cod_forma_coleta_fam,d.dta_entrevista_fam,d.nom_localidade_fam,d.nom_tip_logradouro_fam,d.nom_titulo_logradouro_fam,...,p.ind_dinh_catador_memb,p.ind_dinh_servs_gerais_memb,p.ind_dinh_pede_memb,p.ind_dinh_vendas_memb,p.ind_dinh_outro_memb,p.ind_dinh_nao_resp_memb,p.ind_atend_nenhum_memb,p.ref_cad,p.ref_pbf,renda_per_capita
4,3519071,37372840,07/02/2002,14/04/2025,3,1,14/04/2025,JARDIM MONTE SINAI,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0,0.0
5,3519071,37372840,07/02/2002,14/04/2025,3,1,14/04/2025,JARDIM MONTE SINAI,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0,0.0
10,3519071,47959118,01/03/2002,06/09/2024,3,1,06/09/2024,JARDIM RESIDENCIAL FIRENZE,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0,0.0
11,3519071,47959118,01/03/2002,06/09/2024,3,1,06/09/2024,JARDIM RESIDENCIAL FIRENZE,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0,0.0
12,3519071,47959118,01/03/2002,06/09/2024,3,1,06/09/2024,JARDIM RESIDENCIAL FIRENZE,RUA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/12/2025,202601.0,0.0


In [16]:
len(familias_zero)

13568

In [17]:
familias_zero[["p.dta_nasc_pessoa", "p.cod_sexo_pessoa"]].head()

,p.dta_nasc_pessoa,p.cod_sexo_pessoa
4,30/11/1981,2
5,25/03/2008,1
10,17/10/1981,2
11,12/02/2011,2
12,09/07/1974,1


In [18]:
familias_zero = df[df["d.vlr_renda_total_fam"] == 0].copy()

In [19]:
familias_zero["d.cod_familiar_fam"].nunique()

6259

In [20]:
len(familias_zero)

13568

In [21]:
familias_zero["d.qtd_pessoas_domic_fam"].describe()

count    13168.000000
mean         2.857002
std          1.331958
min          1.000000
25%          2.000000
50%          3.000000
75%          4.000000
max          9.000000
Name: d.qtd_pessoas_domic_fam, dtype: float64

In [22]:
familias_zero["p.cod_sexo_pessoa"].value_counts()

p.cod_sexo_pessoa
2    8197
1    5371
Name: count, dtype: int64

In [23]:
from datetime import datetime

hoje = datetime(2025, 12, 31)

familias_zero["idade"] = (hoje - pd.to_datetime(familias_zero["p.dta_nasc_pessoa"])).dt.days // 365

familias_zero["idade"].describe()

C:\Users\ailto\AppData\Local\Temp\ipykernel_27116\891235491.py:5: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  familias_zero["idade"] = (hoje - pd.to_datetime(familias_zero["p.dta_nasc_pessoa"])).dt.days // 365


count    13568.000000
mean        26.647848
std         19.626744
min          0.000000
25%          9.000000
50%         24.000000
75%         43.000000
max         89.000000
Name: idade, dtype: float64

## 🔎 Análise exploratória — famílias com renda zero

Foram identificadas 6.259 famílias com renda total declarada igual a zero, totalizando 13.568 indivíduos.

A composição média desses domicílios é de aproximadamente 2,85 pessoas, com distribuição consistente (mediana de 3 pessoas por domicílio).

Esse resultado indica que o fenômeno não se trata de inconsistência isolada da base, mas sim de um padrão relevante, possivelmente associado a:

- ausência real de renda monetária
- subdeclaração de renda
- limitações de atualização cadastral
- presença de famílias em situação de extrema vulnerabilidade

Essa condição impacta diretamente o cálculo da variável RT_01, uma vez que tais domicílios são automaticamente classificados abaixo do limite de renda per capita.

## 🧠 Critério de cálculo — RT_01 (referência IPEA)

O indicador RT_01, conforme metodologia do IPEA, é definido como:

> proporção de pessoas que vivem em domicílios com renda domiciliar per capita inferior ou igual a 1/2 salário mínimo.

Diferentemente de uma contagem por famílias, o cálculo é realizado em nível populacional, considerando o número de indivíduos residentes em cada domicílio.

Assim, domicílios com maior número de moradores possuem maior peso no indicador.

Formalmente:

RT_01 = (pessoas em domicílios com renda per capita ≤ 1/2 SM) / (total de pessoas)

No presente projeto (IVS-H), o cálculo é realizado com base na população cadastrada no CadÚnico, constituindo uma aproximação operacional do indicador.

## ⚠️ Interpretação do RT_01 no contexto do CadÚnico

O cálculo do RT_01 neste projeto é realizado com base na população cadastrada no CadÚnico, e não sobre o total da população do município.

Dessa forma, o indicador deve ser interpretado como uma medida da vulnerabilidade relativa dentro do universo das famílias cadastradas em políticas públicas.

O valor obtido não é diretamente comparável ao RT_01 oficial do IPEA, que utiliza dados censitários do IBGE para representar toda a população.

Espera-se, portanto, que o resultado aqui encontrado apresente níveis mais elevados de vulnerabilidade, em função do viés de seleção da base.

## 📌 Referência — cálculo do RT_01

### 🔎 Fórmula oficial (IPEA)

\[
RT_{01}^{IPEA} = \frac{\text{pessoas no Censo com renda ≤ 1/2 SM}}{\text{população total do Censo}}
\]

> O indicador oficial utiliza dados censitários, garantindo representatividade de toda a população.

---

## 🧠 3) O que está sendo construído (IVS-H comEstamos c

Você está construindo uma aproximação baseada em dados administrativos:

\[
RT_{01}^{CadÚnico} = \frac{\text{pessoas vulneráveis dentro do CadÚnico}}{\text{total de pessoas do CadÚnico}}
\]

> ⚠️ Importante: este cálculo não representa a população total do município, mas sim o subconjunto cadastrado, com viés para populações de maior vulnerabilidade.

---

## 🎯 Interpretação

- O RT_01 (CadÚnico) tende a ser **superestimado** em relação ao RT_01 oficial
- Ainda assim, é **extremamente útil** para:
  - análise territorial
  - priorização de políticas públicas
  - leitura de pressão social (IPST-H)

In [24]:
# critério: renda per capita <= 1/2 salário mínimo (R$ 706 em dez/2025)

limite = 706

df["renda_per_capita"] = df["d.vlr_renda_total_fam"] / df["d.qtd_pessoas_domic_fam"]

# identificar pessoas em situação de vulnerabilidade
df["rt01_flag"] = df["renda_per_capita"] <= limite

# cálculo do indicador
rt01 = df["rt01_flag"].mean()

f"{rt01 * 100:.2f}%"
print(f"RT_01 (proporção): {rt01:.4f}")
print(f"RT_01 (percentual): {rt01 * 100:.2f}%")

RT_01 (proporção): 0.6017
RT_01 (percentual): 60.17%


## 🧠 Interpretação do indicador — RT_01 (CadÚnico)

O resultado obtido para a variável **RT_01** indica que aproximadamente:

> **60% das pessoas cadastradas no CadÚnico em Hortolândia possuem renda domiciliar per capita inferior ou igual a 1/2 salário mínimo (R$ 706 em dez/2025).**

---

### 📊 Leitura do resultado

Esse valor não representa apenas um cálculo técnico, mas uma evidência clara da estrutura social observada na base:

- Aproximadamente **6 em cada 10 pessoas** encontram-se em situação de baixa renda
- O dado confirma que o CadÚnico está efetivamente captando a população em maior vulnerabilidade
- Ao mesmo tempo, revela uma **alta concentração de renda baixa dentro do universo cadastrado**

---

### ⚠️ Interpretação metodológica

É importante destacar que:

- O resultado refere-se **exclusivamente à população cadastrada no CadÚnico**
- Não representa a totalidade da população do município
- Trata-se, portanto, de um indicador com viés intencional para populações de maior vulnerabilidade

---

### 🎯 Implicações para o projeto IVS-H

Esse achado possui relevância direta para o desenvolvimento do Atlas Social:

- Reforça a consistência do uso do CadÚnico como base inicial
- Indica forte presença da dimensão **Renda e Trabalho** na vulnerabilidade local
- Oferece um ponto de partida robusto para análises territoriais futuras

---

### 🧭 Aplicação prática

Este resultado é particularmente útil para:

- construção de narrativas executivas
- elaboração de relatórios técnicos
- apresentações institucionais (ex: Secretaria de Inclusão)
- fundamentação de políticas públicas focalizadas

---

> **Síntese:** O indicador RT_01 evidencia que a vulnerabilidade econômica é um componente central da realidade social observada no CadÚnico em Hortolândia.

In [25]:
# criar faixas de tamanho de família
df["faixa_tamanho_familia"] = pd.cut(
    df["d.qtd_pessoas_domic_fam"],
    bins=[0, 2, 4, 6, 10],
    labels=["1-2 pessoas", "3-4 pessoas", "5-6 pessoas", "7+ pessoas"]
)

# calcular RT_01 por faixa
rt01_por_faixa = df.groupby("faixa_tamanho_familia")["rt01_flag"].mean() * 100

rt01_por_faixa

C:\Users\ailto\AppData\Local\Temp\ipykernel_27116\1226604040.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  rt01_por_faixa = df.groupby("faixa_tamanho_familia")["rt01_flag"].mean() * 100


faixa_tamanho_familia
1-2 pessoas    45.199237
3-4 pessoas    69.009557
5-6 pessoas    76.382287
7+ pessoas     86.131387
Name: rt01_flag, dtype: float64

## 🧠 Nota metodológica — unidade de análise do RT_01

A variável **RT_01** é definida a partir da renda domiciliar per capita, sendo, portanto, uma **condição do domicílio**, e não do indivíduo isoladamente.

Isso implica que:

- todas as pessoas pertencentes a um mesmo domicílio compartilham a mesma classificação (vulnerável ou não vulnerável)
- não há distinção intra-familiar quanto à condição de renda

---

### 📊 Forma de cálculo adotada neste notebook

Apesar da definição domiciliar, a operacionalização foi realizada no nível de **registros individuais (pessoas)**, uma vez que a base do CadÚnico está estruturada dessa forma.

Assim:

- cada pessoa herda a condição do domicílio ao qual pertence
- o cálculo do indicador é feito sobre o total de pessoas

---

### 🎯 Interpretação correta

O resultado obtido deve ser interpretado como:

> **percentual de pessoas que vivem em domicílios com renda per capita inferior ou igual a 1/2 salário mínimo**

---

### ⚠️ Implicações

- famílias maiores têm maior peso no indicador, pois possuem mais indivíduos
- o resultado reflete a **distribuição da vulnerabilidade na população cadastrada**, e não apenas entre domicílios

---

### 🧭 Observação

Uma análise complementar poderá ser realizada futuramente no nível de **domicílios (famílias)**, permitindo comparação entre:

- proporção de famílias vulneráveis
- proporção de pessoas em situação de vulnerabilidade

---

> **Síntese:** A vulnerabilidade é definida no nível do domicílio, mas mensurada, nesta etapa, a partir dos indivíduos que o compõem.